<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Processing/colab_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER + DATE/TIME for the GeoRAG corpus
Upload `DATA.json` (or a JSON/JSONL file containing text records). The notebook runs the known BERT baseline for PER/ORG/LOC/MISC and spaCy for DATE/TIME. DATE/TIME are text mentions, not validated event dates.

Start with `LIMIT = 2`, inspect the outputs, then set `LIMIT = None` for the full corpus.


In [ ]:
%pip -q install transformers spacy torch
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import json
import subprocess
from pathlib import Path
import transformers
import spacy
import torch

הגדרת תיקיות

In [ ]:
repo_dir = Path("/content/Corpus2GeoRAG_Pacific")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main",
         "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git",
         str(repo_dir)],
        check=True,
    )

In [ ]:
data_dir = repo_dir / "Data"
json_files = list(data_dir.glob("*.json"))
if len(json_files) != 1:
    raise ValueError(f"ציפיתי לקובץ JSON אחד בתיקיית Data; נמצאו: {[p.name for p in json_files]}")

קריאה לDATA

In [ ]:
input_path = json_files[0]
output_dir = data_dir / "TMP_NER"

with input_path.open(encoding="utf-8") as f:
    data = json.load(f)

print(f"נטען: {input_path.name} — {len(data)} מסמכים")
print(f"פלט יישמר ב: {output_dir}")

נטען: DATA.json — 428 מסמכים
פלט יישמר ב: /content/Corpus2GeoRAG_Pacific/Data/TMP_NER


הצגת 3 הKEYS הראשונים

In [ ]:
print(list(data.keys())[:3]) # שלוש הכותרות הראשונות

['14th Indian Infantry Division', '1939–1940 Winter Offensive', '1943 Cairo Declaration']


הגרלת 100 פסקאות

In [ ]:
import json
import random
import re

SEED = 42
SAMPLE_SIZE = 100

paragraphs = []

for title, text in data.items():
    separators = list(re.finditer(r"\n[ \t]*\n+", text))
    starts = [0] + [m.end() for m in separators]
    ends = [m.start() for m in separators] + [len(text)]

    for index, (left, right) in enumerate(zip(starts, ends)):
        chunk = text[left:right]
        paragraph = chunk.strip()

        if not paragraph or re.fullmatch(r"=+\s*.*?\s*=+", paragraph):
            continue

        start = left + len(chunk) - len(chunk.lstrip())
        end = left + len(chunk.rstrip())

        paragraphs.append({
            "article_title": title,
            "paragraph_index": index,
            "start": start,
            "end": end,
            "text": text[start:end],
        })

paragraphs.sort(key=lambda p: (p["article_title"], p["start"]))

if SAMPLE_SIZE > len(paragraphs):
    raise ValueError(f"ביקשת {SAMPLE_SIZE} פסקאות, אך נמצאו רק {len(paragraphs)}")

sample = random.Random(SEED).sample(paragraphs, SAMPLE_SIZE)

output_dir.mkdir(parents=True, exist_ok=True)
sample_path = output_dir / f"sample_{SAMPLE_SIZE}_paragraphs_seed{SEED}.json"

result = {
    "source_file": input_path.name,
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "paragraphs": sample,
}

with sample_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"נשמרו {len(sample)} פסקאות מתוך {len(paragraphs)}")
print(sample_path)

נשמרו 100 פסקאות מתוך 9496
/content/Corpus2GeoRAG_Pacific/Data/TMP_NER/sample_100_paragraphs_seed42.json


העלאה לGIT

SecretNotFoundError: Secret GITHUB_TOKEN does not exist.

In [ ]:
ניסיון